# MiniMax H3 Singularity Pruned · Kaggle T4 × 2

**Versão: `singularity-working-base-r1`.** Adaptação do notebook que você enviou e que já abre
o ComfyUI. A instalação direta, uso de `/kaggle/temp`, divisão do encoder na segunda T4,
túnel e interface foram preservados. Não usa `venv` nem `ensurepip`.

Mudanças: checkpoint **Singularity REF2VA Pruned INT8**, LoRA REF2V **4step v0.1**,
sampler nativo Euler + beta em uma passagem, com 6 passos por padrão.
O grafo usa somente uma LoRA de velocidade e corrige os passos realmente conectados
ao scheduler; não apenas o valor de um widget que seria sobrescrito por uma conexão.

Fonte do autor: [AIGC-Singularity/Minimax-h3_Singularity](https://github.com/AIGC-Singularity/Minimax-h3_Singularity).
Pesos: [WarmBloodAban/Minimax-h3_Singularity](https://huggingface.co/WarmBloodAban/Minimax-h3_Singularity).
O nome exato do arquivo pruned é conferido no catálogo do autor durante a execução,
respeitando maiúsculas e subpastas. Nunca troca silenciosamente pelo REF2VA oficial ou pelo Singularity completo.

**Uso:** habilite T4 × 2 e Internet, execute em ordem e abra
**Workflows → `singularity_pruned_t4x2`**. Envie sua imagem pelo LoadImage.
Não precisa anexar imagem antes de abrir a interface. Pesos já anexados são reutilizados;
os que faltam vão para `/kaggle/temp`, não para os ~20 GB de `/kaggle/working`.
O espaço real é conferido antes do download.

Mantidos: Qwen NVFP4, VAE de vídeo INT8 e VAE de áudio FP32, já usados no seu notebook.
Não baixa os LoRAs de 8 passos / EMA nem o upscaler para esta execução de uma passagem.
O `MiniMaxH3DualClockSamplerT8` do JSON antigo não tem origem identificada; esta adaptação usa
os nós nativos de sampling, como o workflow público do autor, sem inventar um plugin T8.

O catálogo Hugging Face e a inferência com os pesos serão verificados no Kaggle.
Validação local do grafo não comprova a qualidade visual nem que o modelo cabe numa T4.


## ① Configuração

Os padrões abaixo são pensados para **Reference-to-Video em T4×2**.


In [ ]:
# =========================
# CONFIGURAÇÃO PRINCIPAL
# =========================

WEIGHT_PROFILE = "R2V"

# ComfyUI recente é recomendado para MiniMax H3 R2V.
# "master" evita incompatibilidade com os nós do template oficial atual.
COMFYUI_TAG = "master"

INSTALL_DUAL_ENCODER_NODE = True
INSTALL_COMFYUI_MANAGER = True
INSTALL_H3_PROMPT_BUILDER = False   # deixe False se não precisar do plugin de prompts

# Tenta habilitar kernels mais rápidos quando o driver do Kaggle permitir.
TRY_CU130_TORCH = True

# Gera também workflow R2V Turbo 4-step.
USE_TURBO_LORA = True

# "cloudflared" não exige token.
TUNNEL_METHOD = "cloudflared"
PORT = 8188

# O repositório Comfy-Org/MiniMax-H3 normalmente não exige token.
# Se em algum momento passar a exigir aceite/token, adicione HF_TOKEN em Kaggle Secrets.
HF_TOKEN = None

# ---------- ARMAZENAMENTO ----------
# "auto" = procura os arquivos em /kaggle/input; baixa apenas o que estiver faltando.
# "attached_only" = exige que todos os pesos já estejam anexados via Kaggle Model.
# "download" = ignora /kaggle/input e baixa para /kaggle/temp.
MODEL_SOURCE = "auto"

# Dica opcional para acelerar a busca por um Model anexado.
# Pode deixar vazio; o notebook procura recursivamente em /kaggle/input.
KAGGLE_MODEL_HINT = ""

# Dataset privado contendo suas referências.
# Exemplo: se aparecer /kaggle/input/meu-r2v-assets, use "meu-r2v-assets".
ASSET_DATASET_HINT = ""

# Publicação opcional dos pesos baixados para a cota "Modelos Privados".
# Deixe False na execução normal.
PUBLISH_WEIGHTS_TO_KAGGLE_MODEL = False

# Ajuste se quiser usar a célula de publicação.
# Formato exigido pelo kagglehub:
# <usuario>/<modelo>/<framework>/<variation>
KAGGLE_MODEL_HANDLE = "eliascordeiro/minimax-h3-r2v/pyTorch/comfyui-r2v"

if MODEL_SOURCE not in ("auto", "attached_only", "download"):
    raise ValueError('MODEL_SOURCE deve ser "auto", "attached_only" ou "download"')
if TUNNEL_METHOD not in ("cloudflared", "cloudflare-named", "ngrok", "auto"):
    raise ValueError("TUNNEL_METHOD inválido")


# ---------- SINGULARITY PRUNED ----------
NOTEBOOK_VERSION = "singularity-working-base-r1"
SINGULARITY_REPO = "WarmBloodAban/Minimax-h3_Singularity"
SINGULARITY_REVISION = "main"  # será resolvida para um commit no catálogo
SINGULARITY_FILE = ""  # automático: somente REF2VA + Pruned + INT8 + v1.3
SINGULARITY_LOCAL_PATH = ""  # opcional: caminho exato de um peso anexado
SINGULARITY_STEPS = 6
SINGULARITY_MEGAPIXELS = 0.2  # mesmo tamanho inicial do notebook que funciona; depois 0.3/0.4
USE_TURBO_LORA = True  # Singularity usa a REF2V 4step v0.1, sem empilhar outras LoRAs
print("Notebook:", NOTEBOOK_VERSION)


## ② Diagnóstico: GPU / RAM / disco

In [ ]:
import json
import os
import platform
import shutil
import subprocess
import sys
import time
from pathlib import Path

try:
    import torch
except ImportError as exc:
    raise RuntimeError("PyTorch não está disponível no ambiente.") from exc

ON_KAGGLE = Path("/kaggle").exists()

WORK_ROOT = Path("/kaggle/temp/h3_r2v_t4x2") if ON_KAGGLE else Path("./h3_r2v_t4x2").resolve()
COMFY_HOME = WORK_ROOT / "ComfyUI"
CACHE_HOME = WORK_ROOT / "hf_cache"
MODEL_CACHE = WORK_ROOT / "model_cache"
GEN_DIR = WORK_ROOT / "gen"
TMP_DIR = WORK_ROOT / "tmp"
FINAL_DIR = Path("/kaggle/working") if ON_KAGGLE else WORK_ROOT / "dist"
LOG_FILE = FINAL_DIR / "h3_r2v_comfy.log"

for d in (WORK_ROOT, CACHE_HOME, MODEL_CACHE, GEN_DIR, TMP_DIR, FINAL_DIR):
    d.mkdir(parents=True, exist_ok=True)

def to_gib(num):
    return num / 1024**3

def log_tail(n=80):
    if not LOG_FILE.exists():
        return "(sem log)"
    lines = LOG_FILE.read_text(encoding="utf-8", errors="replace").splitlines()
    return "\n".join(lines[-n:])

print(f"Python {sys.version.split()[0]} | {platform.system()} {platform.release()}")
print(f"PyTorch {torch.__version__} | CUDA disponível: {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise RuntimeError("GPU não detectada. No Kaggle, selecione Accelerator = GPU T4 x2.")

GPU_COUNT = torch.cuda.device_count()
for i in range(GPU_COUNT):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU{i}: {p.name} · {to_gib(p.total_memory):.1f} GiB")

if GPU_COUNT < 2:
    print("AVISO: só uma GPU detectada. O workflow ainda pode rodar, mas com maior risco de OOM.")
else:
    print("OK: duas GPUs detectadas. O Qwen/CLIP será direcionado para cuda:1.")

try:
    import psutil
    print(f"RAM: {to_gib(psutil.virtual_memory().total):.1f} GiB")
except Exception:
    pass

free_gib = to_gib(shutil.disk_usage(WORK_ROOT).free)
print(f"Disco temporário livre: {free_gib:.1f} GiB")
print("Observação: se os pesos vierem de um Kaggle Model anexado, eles NÃO precisam ser copiados para /kaggle/working.")

subprocess.run(["nvidia-smi"], check=False)


## ③ Instalar ComfyUI e dependências

Esta versão usa `master` por padrão para manter compatibilidade com os nós R2V atuais.


In [ ]:
import subprocess
import sys
from pathlib import Path

def sh(cmd, cwd=None):
    print("$", " ".join(map(str, cmd)))
    subprocess.run(list(map(str, cmd)), cwd=cwd, check=True)

REPO_URL = "https://github.com/comfyanonymous/ComfyUI.git"

if not (COMFY_HOME / ".git").exists():
    try:
        sh(["git", "clone", "--depth", "1", "--branch", COMFYUI_TAG, REPO_URL, str(COMFY_HOME)])
    except subprocess.CalledProcessError:
        print(f"稳定版 {COMFYUI_TAG} 拉取失败，退回 master 分支。")
        sh(["git", "clone", "--depth", "1", REPO_URL, str(COMFY_HOME)])
else:
    print("ComfyUI 已存在，跳过克隆。")

sh([sys.executable, "-m", "pip", "install", "-q", "-r", str(COMFY_HOME / "requirements.txt")])
sh([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "huggingface_hub>=1.23.0,<2.0",
    "requests",
    "psutil",
    "pyngrok",
    "kagglehub"
])
subprocess.run([sys.executable, "-m", "pip", "cache", "purge"], check=False)

COMFY_COMMIT = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"], cwd=COMFY_HOME, text=True
).strip()
print("ComfyUI commit:", COMFY_COMMIT)

# ── 升级 PyTorch 到 cu130，启用 ComfyUI 快速量化内核 ──
# Kaggle conda 预装的 torch 会占用 import 路径，pip install 覆盖不了。
# 解法：先 pip uninstall 彻底卸掉旧的，再从 cu130 index 全新安装（--no-cache-dir 确保不读缓存）。
def _driver_cuda13_ok():
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"],
            capture_output=True, text=True, timeout=30).stdout.strip().splitlines()
        major = int(out[0].split(".")[0]) if out else 0
        return major >= 580
    except Exception:
        return False

def _check_subprocess_torch():
    """检查子进程（ComfyUI 运行环境）实际加载的 torch/cuda 版本。"""
    try:
        r = subprocess.run(
            [sys.executable, "-c",
             "import torch; print(f'{torch.__version__}|{torch.version.cuda}')"],
            capture_output=True, text=True, timeout=60
        )
        if r.returncode == 0:
            ver, cuda = r.stdout.strip().split("|")
            return ver, cuda
    except Exception:
        pass
    return "?", "?"

if TRY_CU130_TORCH and _driver_cuda13_ok():
    _sub_ver, _sub_cuda = _check_subprocess_torch()
    if _sub_cuda.startswith("13."):
        print(f"\u2714 子进程 torch 已是 cu130（{_sub_ver} | cuda {_sub_cuda}），无需升级。")
        _new_ver, _new_cuda = _sub_ver, _sub_cuda
    else:
        print(f"驱动支持 CUDA 13；当前子进程 torch {_sub_ver} | cuda {_sub_cuda} \u2192 升级到 cu130 \u2026")
        print("  1/4 卸载旧 torch（解决 conda 路径优先级问题）\u2026")
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y",
                        "torch", "torchvision", "torchaudio"],
                       capture_output=True, timeout=120)
        print("  2/4 安装 cu130 torch（--no-cache-dir 确保不读旧缓存）\u2026")
        sh([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
            "--index-url", "https://download.pytorch.org/whl/cu130",
            "torch", "torchvision", "torchaudio"])
        print("  3/4 校验子进程是否真的加载了 cu130 \u2026")
        _new_ver, _new_cuda = _check_subprocess_torch()
        print(f"  3/4 校验：子进程 torch {_new_ver} | cuda {_new_cuda}")
        if not _new_cuda.startswith("13."):
            print(f"\u26a0\ufe0f  子进程仍为 cu{_new_cuda} \u2014\u2014 conda 路径优先级太强。")
            print("  尝试 --force-reinstall 覆盖到 conda site-packages \u2026")
            import torch as _t
            _conda_site = str(Path(_t.__file__).parent.parent)
            try:
                subprocess.run([sys.executable, "-m", "pip", "install",
                                "--force-reinstall", "--no-deps", "--no-cache-dir",
                                "--index-url", "https://download.pytorch.org/whl/cu130",
                                "--target", _conda_site,
                                "torch", "torchvision", "torchaudio"],
                               check=True, timeout=300, capture_output=True, text=True)
                _new_ver, _new_cuda = _check_subprocess_torch()
                print(f"  覆盖后：torch {_new_ver} | cuda {_new_cuda}")
            except Exception as _exc2:
                print(f"  覆盖安装失败：{_exc2}")
    # 4/4 重建 comfy_kitchen（torch 升级到 cu130 后，必须重装使其 CUDA 后端生效）
    if _new_cuda.startswith("13."):
        print("  4/4 重建 comfy_kitchen（torch 升级后 C 扩展需要重新编译）\u2026")
        subprocess.run([sys.executable, "-m", "pip", "install", "--force-reinstall", "--no-cache-dir",
                        "comfy-kitchen"], capture_output=True, timeout=300)
        _ck_verify = subprocess.run([sys.executable, "-c",
            "import comfy_kitchen as ck; import torch; "
            "print(f'ck ok | torch {torch.__version__} | cuda {torch.version.cuda}'); "
            "print('backends:', ck.list_backends())"],
            capture_output=True, text=True, timeout=60)
        if _ck_verify.returncode == 0:
            print("  \u2714 comfy_kitchen 重建完成：", _ck_verify.stdout.strip())
        else:
            print("  \u26a0\ufe0f  comfy_kitchen 重建后仍有问题：", _ck_verify.stderr[:300])
            print("     不影响使用，ComfyUI 会退到慢速路径。")
    else:
        print("  \u26a0\ufe0f  torch 非 cu130，跳过 comfy_kitchen 重建。")
        print("  Use o workflow R2V Turbo 4-step para reduzir bastante o tempo de amostragem.")
else:
    print("跳过 cu130 升级（TRY_CU130_TORCH=False 或驱动 < 580）。")


In [ ]:
# instalador do huggingface hub pois o de cima falhou mas já foi concertado
!python -m pip install -q -U "huggingface_hub>=1.23.0,<2.0"

import huggingface_hub
print("huggingface_hub:", huggingface_hub.__version__)

## ③.5 Plugins opcionais

O notebook original instala ComfyUI Manager e H3 Prompt Builder.  
Aqui o **Manager continua habilitado**; o H3 Prompt Builder vem desabilitado por padrão porque não é necessário para Reference-to-Video.


In [ ]:
# ══ 兜底：前置格子没跑也能独立运行 ══
if "COMFY_HOME" not in globals():
    from pathlib import Path as _P
    WORK_ROOT = _P("/kaggle/temp/h3_t4x2") if _P("/kaggle").exists() else _P("./h3_t4x2").resolve()
    COMFY_HOME = WORK_ROOT / "ComfyUI"
if "INSTALL_COMFYUI_MANAGER" not in globals():
    INSTALL_COMFYUI_MANAGER = True
if "INSTALL_H3_PROMPT_BUILDER" not in globals():
    INSTALL_H3_PROMPT_BUILDER = True

import subprocess
CUSTOM_NODES = COMFY_HOME / "custom_nodes"
CUSTOM_NODES.mkdir(parents=True, exist_ok=True)

def sh(cmd, cwd=None):
    print("$", " ".join(map(str, cmd)))
    subprocess.run(list(map(str, cmd)), cwd=cwd, check=True)

# ── ComfyUI Manager ──
if INSTALL_COMFYUI_MANAGER:
    mgr_dir = CUSTOM_NODES / "ComfyUI-Manager"
    if not mgr_dir.exists():
        print("安装 ComfyUI Manager …")
        sh(["git", "clone", "--depth", "1",
            "https://github.com/ltdrdata/ComfyUI-Manager.git", str(mgr_dir)])
    else:
        print("ComfyUI Manager 已存在，跳过。")
    print("✔ ComfyUI Manager 就绪（网页里点 Manager 按钮即可使用）")

# ── H3 Prompt Builder ──
if INSTALL_H3_PROMPT_BUILDER:
    h3pb_dir = CUSTOM_NODES / "ComfyUI-H3-Prompt-Builder"
    if not h3pb_dir.exists():
        print("安装 H3 Prompt Builder …")
        sh(["git", "clone", "--depth", "1",
            "https://github.com/jiel71365-commits/ComfyUI-H3-Prompt-Builder.git",
            str(h3pb_dir)])
        # 仓库里插件源码在 comfyui-h3-prompt-builder/ 子目录，提上来
        sub = h3pb_dir / "comfyui-h3-prompt-builder"
        if sub.exists():
            import shutil as _sh
            for item in sub.iterdir():
                dst = h3pb_dir / item.name
                if dst.exists():
                    if dst.is_dir():
                        _sh.rmtree(dst)
                    else:
                        dst.unlink()
                _sh.move(str(item), str(dst))
            _sh.rmtree(sub)
            print(f"  插件文件已提升到 {h3pb_dir}")
    else:
        print("H3 Prompt Builder 已存在，跳过。")

    # 自动写入 DeepSeek API Key（从 Kaggle Secrets 读取）
    cfg_path = h3pb_dir / "config.json"
    cfg = {
        "base_url": "https://api.deepseek.com/chat/completions",
        "model": "deepseek-v4-flash",
        "api_key": "",
        "temperature": 0.4,
        "max_tokens": 32768,
        "manju_temperature": 0.2,
        "thinking_disabled": True,
        "request_timeout": 240,
        "reviewer_temperature": 0.1,
        "max_review_rounds": 8,
    }
    # 如果已有 config.json，读取合并
    if cfg_path.exists():
        try:
            existing = json.loads(cfg_path.read_text(encoding="utf-8-sig"))
            cfg.update(existing)
        except Exception:
            pass

    # 尝试从 Kaggle Secrets 读取 API Key
    try:
        from kaggle_secrets import UserSecretsClient
        _secrets = UserSecretsClient()
        _key = _secrets.get_secret("DEEPSEEK_API_KEY")
        if _key:
            cfg["api_key"] = _key.strip()
            print("✔ 已从 Kaggle Secrets 读取 DEEPSEEK_API_KEY 并写入 config.json")
        else:
            print("⚠️  Kaggle Secrets 中未找到 DEEPSEEK_API_KEY（提示词节点可在输入框手动填）")
    except Exception:
        print("⚠️  无法读取 Kaggle Secrets（非 Kaggle 环境？可在节点输入框手动填 API Key）")

    cfg_path.write_text(json.dumps(cfg, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"✔ H3 Prompt Builder 就绪（config: {cfg_path}）")
    print("  节点分类：MiniMax H3 → H3 Prompt Builder；MiniMax H3 / 漫剧 → 漫剧 6 节点")

print("\n打开 ComfyUI 网页后：")
print("  • H3 Prompt Builder：右键 → MiniMax H3 → H3 Prompt Builder（提示词生成）")
print("  • 漫剧分镜：右键 → MiniMax H3 / 漫剧 → 选择对应节点")
print("  • Manager：网页顶部 Manager 按钮可搜索安装更多插件")

## ④ Nó Dual-T4

Mantemos a ideia mais útil do notebook original:

- **Text Encoder / Qwen → `cuda:1`**
- **Diffusion + VAE → `cuda:0`**

Isso evita tentar manter o Qwen3-VL de ~14.6 GB na mesma T4 usada pelo diffusion model.


In [ ]:
if not INSTALL_DUAL_ENCODER_NODE:
    print("已跳过（INSTALL_DUAL_ENCODER_NODE=False）。")
else:
    NODE_DIR = COMFY_HOME / "custom_nodes" / "t4x2_dual_encoder"
    NODE_DIR.mkdir(parents=True, exist_ok=True)

    NODE_SRC = '''
"""Kaggle T4x2 专用：把文本编码器 / VAE 显式加载到指定显卡。"""

import torch
import folder_paths
import comfy.sd
import comfy.utils


def _devices():
    return [f"cuda:{i}" for i in range(torch.cuda.device_count())] + ["cpu"]


class DualT4TextEncoder:
    """可选设备的 CLIP 加载器：cuda:0 / cuda:1 / cpu。"""

    @classmethod
    def INPUT_TYPES(cls):
        return {
            "required": {
                "clip_name": (folder_paths.get_filename_list("text_encoders"),),
                "load_to": (_devices(),),
            }
        }

    RETURN_TYPES = ("CLIP",)
    FUNCTION = "load"
    CATEGORY = "loaders"

    def load(self, clip_name, load_to):
        clip_type = getattr(comfy.sd.CLIPType, "MINIMAX", None)
        if clip_type is None:
            raise RuntimeError(
                "当前 ComfyUI 版本过旧：MiniMax H3 需要 v0.30.0 及以上。"
            )
        path = folder_paths.get_full_path_or_raise("text_encoders", clip_name)
        options = {
            "load_device": torch.device(load_to),
            "offload_device": torch.device("cpu"),
        }
        clip = comfy.sd.load_clip(
            ckpt_paths=[path],
            embedding_directory=folder_paths.get_folder_paths("embeddings"),
            clip_type=clip_type,
            model_options=options,
        )
        return (clip,)


class DualT4VAELoader:
    """可选设备的 VAE 加载器：把视频/音频 VAE 的编解码挪到第二张卡。"""

    @classmethod
    def INPUT_TYPES(cls):
        return {
            "required": {
                "vae_name": (folder_paths.get_filename_list("vae"),),
                "load_to": (_devices(),),
            }
        }

    RETURN_TYPES = ("VAE",)
    FUNCTION = "load"
    CATEGORY = "loaders"

    def load(self, vae_name, load_to):
        path = folder_paths.get_full_path_or_raise("vae", vae_name)
        sd = comfy.utils.load_torch_file(path)
        vae = comfy.sd.VAE(sd=sd, device=torch.device(load_to))
        return (vae,)


NODE_CLASS_MAPPINGS = {
    "DualT4TextEncoder": DualT4TextEncoder,
    "DualT4VAELoader": DualT4VAELoader,
}
NODE_DISPLAY_NAME_MAPPINGS = {
    "DualT4TextEncoder": "Dual-T4 Text Encoder (Kaggle)",
    "DualT4VAELoader": "Dual-T4 VAE Loader (Kaggle)",
}
'''

    (NODE_DIR / "__init__.py").write_text(NODE_SRC, encoding="utf-8")
    print("已写入自定义节点:", NODE_DIR / "__init__.py")

## ⑥ Workflow Singularity Pruned

Confere o checkpoint no catálogo do autor ou nos pesos anexados e adapta o template REF2VA.
Mantém o loader Dual-T4 do seu notebook. A variante usa uma passagem e a LoRA v0.1 de 4 passos;
6 passos de amostragem é o padrão de teste indicado no workflow original enviado.


In [ ]:
import copy
import json
import os
import shutil
import urllib.request
from pathlib import Path
from huggingface_hub import HfApi

def singularity_pruned_name(filename):
    name = Path(filename).name.lower()
    return (name.endswith('.safetensors') and 'singularity' in name
            and 'ref2va' in name and 'pruned' in name and 'int8' in name and 'v1.3' in name)

def select_singularity_file(filenames, explicit=''):
    if explicit:
        if explicit not in filenames:
            raise FileNotFoundError('Arquivo configurado não existe no catálogo: ' + explicit)
        if not singularity_pruned_name(explicit):
            raise ValueError('O arquivo deve ser Singularity REF2VA Pruned v1.3 INT8.')
        return explicit
    matches = [name for name in filenames if singularity_pruned_name(name)]
    if len(matches) != 1:
        listed = [name for name in filenames if 'singularity' in name.lower() and name.endswith('.safetensors')]
        raise RuntimeError('Não há um único checkpoint Singularity Pruned v1.3 INT8 no catálogo. '
                           'Defina SINGULARITY_FILE com um arquivo publicado ou SINGULARITY_LOCAL_PATH. '
                           'Candidatos: ' + json.dumps(listed, ensure_ascii=False))
    return matches[0]

INPUT_ROOT = Path('/kaggle/input')
SINGULARITY_SOURCE_PATH = None
if SINGULARITY_LOCAL_PATH:
    SINGULARITY_SOURCE_PATH = Path(SINGULARITY_LOCAL_PATH)
    if not SINGULARITY_SOURCE_PATH.is_file() or not singularity_pruned_name(SINGULARITY_SOURCE_PATH.name):
        raise ValueError('SINGULARITY_LOCAL_PATH deve apontar para o arquivo Singularity Pruned v1.3 INT8.')
elif MODEL_SOURCE in ('auto', 'attached_only') and INPUT_ROOT.exists():
    matches = [path for path in INPUT_ROOT.rglob('*.safetensors') if singularity_pruned_name(path.name)]
    if len(matches) > 1:
        raise ValueError('Mais de um Singularity anexado; escolha SINGULARITY_LOCAL_PATH.')
    if matches:
        SINGULARITY_SOURCE_PATH = matches[0]

SINGULARITY_PUBLISHED_SIZE = None
if SINGULARITY_SOURCE_PATH is not None:
    SINGULARITY_FILE = SINGULARITY_SOURCE_PATH.name
elif MODEL_SOURCE == 'attached_only':
    raise FileNotFoundError('Anexe o Singularity Pruned como Kaggle Model/Dataset ou use MODEL_SOURCE="auto".')
else:
    # Só consulta o catálogo; nenhum peso é baixado nesta célula.
    catalog = HfApi(token=HF_TOKEN or os.environ.get('HF_TOKEN')).model_info(
        SINGULARITY_REPO, revision=SINGULARITY_REVISION, files_metadata=True,
    )
    SINGULARITY_FILE = select_singularity_file([item.rfilename for item in catalog.siblings], SINGULARITY_FILE)
    SINGULARITY_REVISION = catalog.sha
    SINGULARITY_PUBLISHED_SIZE = next(item.size for item in catalog.siblings if item.rfilename == SINGULARITY_FILE)

SINGULARITY_NAME = Path(SINGULARITY_FILE).name
print('Checkpoint selecionado:', SINGULARITY_REPO, SINGULARITY_FILE)
print('Revisão:', SINGULARITY_REVISION if SINGULARITY_SOURCE_PATH is None else 'arquivo anexado')
if not USE_TURBO_LORA:
    raise ValueError('Mantenha USE_TURBO_LORA=True: esta variante usa a REF2V 4step v0.1.')
if not isinstance(SINGULARITY_STEPS, int) or not 4 <= SINGULARITY_STEPS <= 12:
    raise ValueError('SINGULARITY_STEPS deve ficar entre 4 e 12; comece com 6.')

TPL_URL = 'https://raw.githubusercontent.com/Comfy-Org/workflow_templates/b4119d0fcecf78a376e087305ec4f485c66a03f5/templates/video_minimax_h3_r2v.json'
with urllib.request.urlopen(TPL_URL, timeout=90) as response:
    wf = json.load(response)
LORA_FILE = 'minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors'

def patch_singularity_workflow(workflow, filename, steps, megapixels):
    graph = copy.deepcopy(workflow)
    nodes = {node['id']: node for node in graph['nodes']}
    # Pinned official template: one LoRA/switch pair, no appended second LoRA.
    required = {127: 'UNETLoader', 128: 'CLIPLoader', 145: 'LoraLoaderModelOnly',
                146: 'PrimitiveBoolean', 143: 'PrimitiveInt', 144: 'PrimitiveInt',
                124: 'BasicScheduler', 123: 'KSamplerSelect', 141: 'ComfySwitchNode'}
    for identifier, kind in required.items():
        if identifier not in nodes or nodes[identifier]['type'] != kind:
            raise RuntimeError('Template não corresponde ao grafo conferido: ' + str(identifier))
    nodes[127]['widgets_values'] = [filename, 'default']
    nodes[127]['title'] = 'Singularity REF2VA Pruned v1.3 INT8'
    nodes[127].setdefault('properties', {})['models'] = []
    for node in nodes.values():
        kind = node['type']
        if kind == 'CLIPLoader':
            node['type'] = 'DualT4TextEncoder'
            node['widgets_values'] = [node['widgets_values'][0], 'cuda:1']
            node['widgets_values_named'] = {'clip_name': node['widgets_values'][0], 'load_to': 'cuda:1'}
            node['title'] = 'Qwen3-VL → cuda:1'
            node.setdefault('properties', {})['Node name for S&R'] = node['type']
        elif kind == 'VAELoader':
            node['type'] = 'DualT4VAELoader'
            node['widgets_values'] = [node['widgets_values'][0], 'cuda:0']
            node['widgets_values_named'] = {'vae_name': node['widgets_values'][0], 'load_to': 'cuda:0'}
            node['title'] = 'VAE → cuda:0'
            node.setdefault('properties', {})['Node name for S&R'] = node['type']
        elif kind == 'ResolutionSelector':
            node['widgets_values'][1] = megapixels
            node.setdefault('widgets_values_named', {})['megapixels'] = megapixels
        elif kind == 'MiniMaxH3ReferenceToVideo':
            node['widgets_values'][4] = 'match'
            node.setdefault('widgets_values_named', {})['ref_image_size'] = 'match'
    nodes[145]['widgets_values'] = [LORA_FILE, 1.0]
    nodes[145]['widgets_values_named'] = {'lora_name': LORA_FILE, 'strength_model': 1.0}
    nodes[146]['widgets_values'] = [True]
    nodes[146]['widgets_values_named'] = {'value': True}
    for identifier in [143, 144]:
        nodes[identifier]['widgets_values'] = [steps, 'fixed']
        nodes[identifier]['widgets_values_named'] = {'value': steps}
    nodes[124]['widgets_values'] = ['beta', steps, 1.0]
    nodes[124]['widgets_values_named'] = {'scheduler': 'beta', 'steps': steps, 'denoise': 1.0}
    nodes[123]['widgets_values'] = ['euler']
    nodes[123]['widgets_values_named'] = {'sampler_name': 'euler'}
    # Route sigmas and guider through the same selected model.
    link = next(link for link in graph['links'] if link[0] == 252)
    link[1:3] = [141, 0]
    nodes[127]['outputs'][0]['links'].remove(252)
    nodes[141]['outputs'][0]['links'].append(252)
    graph['extra']['singularity_model'] = filename
    graph['extra']['singularity_steps'] = steps
    return graph

wf = patch_singularity_workflow(wf, SINGULARITY_NAME, SINGULARITY_STEPS, SINGULARITY_MEGAPIXELS)
WF_DIR = COMFY_HOME / 'user' / 'default' / 'workflows'
WF_DIR.mkdir(parents=True, exist_ok=True)
# Preserve aliases referenced by the existing server/tunnel cell.
for name in ['singularity_pruned_t4x2', 'dual_t4_minimax_h3_r2v', 'dual_t4_minimax_h3_r2v_turbo4']:
    path = WF_DIR / (name + '.json')
    path.write_text(json.dumps(wf, ensure_ascii=False), encoding='utf-8')
    shutil.copy2(path, FINAL_DIR / path.name)
print('Abra Workflows → singularity_pruned_t4x2')
print('Uma LoRA REF2V 4step v0.1; uma passagem;', SINGULARITY_STEPS, 'passos reais; Euler + beta.')


## ⑦ Pesos para Singularity

O diffusion model vem do autor do Singularity. Encoder, VAEs e LoRA de 4 passos vêm
de Comfy-Org/MiniMax-H3. O REF2VA pruned oficial **não** é baixado como substituto.
Reutiliza os pesos já presentes em `/kaggle/input` ou no cache temporário.


In [ ]:
import os
import json
import struct
import shutil
from pathlib import Path
from huggingface_hub import HfApi, hf_hub_download

OFFICIAL_REPO = 'Comfy-Org/MiniMax-H3'
SPECS = [{'target': 'diffusion_models/' + SINGULARITY_NAME, 'repo': SINGULARITY_REPO,
          'file': SINGULARITY_FILE, 'revision': SINGULARITY_REVISION, 'local': SINGULARITY_SOURCE_PATH}]
for rel in ['text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors',
            'vae/minimax_h3_video_vae_int8_convrot.safetensors',
            'vae/minimax_h3_audio_vae_fp32.safetensors',
            'loras/minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors']:
    SPECS.append({'target': rel, 'repo': OFFICIAL_REPO, 'file': rel, 'revision': 'main', 'local': None})
REQUIRED_FILES = [spec['target'] for spec in SPECS]
MODELS_DIR = COMFY_HOME / 'models'
MODEL_CACHE.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(CACHE_HOME)
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'

def check_weight(path, expected_size=None):
    path = Path(path)
    size = path.stat().st_size
    if expected_size is not None and size != expected_size:
        raise ValueError('Peso incompleto ou de outra versão: ' + path.name)
    with path.open('rb') as handle:
        raw = handle.read(8)
        if len(raw) != 8:
            raise ValueError('Peso truncado: ' + path.name)
        header_size = struct.unpack('<Q', raw)[0]
        if not 2 <= header_size <= min(size - 8, 100 * 1024**2):
            raise ValueError('Cabeçalho safetensors inválido: ' + path.name)
        header = json.loads(handle.read(header_size))
    tensors = [value for key, value in header.items() if key != '__metadata__']
    if not tensors or max(t['data_offsets'][1] for t in tensors) != size - 8 - header_size:
        raise ValueError('Dados safetensors incompletos: ' + path.name)
    return size

def safe_link(src, dst):
    src, dst = Path(src).resolve(), Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.is_symlink():
        if dst.resolve() == src:
            return
        dst.unlink()
    elif dst.exists():
        if dst.resolve() != src:
            raise FileExistsError('Não sobrescreverei um peso existente: ' + str(dst))
        return
    dst.symlink_to(src)

token = HF_TOKEN or os.environ.get('HF_TOKEN')
if not token:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        token = None
sources, missing = {}, []
for spec in SPECS:
    target = MODELS_DIR / spec['target']
    cached = MODEL_CACHE / spec['target']
    local = spec['local']
    if local is None and MODEL_SOURCE in ('auto', 'attached_only') and INPUT_ROOT.exists():
        matches = list(INPUT_ROOT.rglob(Path(spec['target']).name))
        if len(matches) > 1:
            raise ValueError('Peso duplicado nos inputs: ' + spec['target'])
        if matches:
            local = matches[0]
    if local is None and target.is_file():
        local = target.resolve()
    if local is None and cached.is_file():
        local = cached
    if local is not None:
        check_weight(local)
        safe_link(local, target)
        sources[spec['target']] = ('Kaggle Input' if '/kaggle/input/' in str(local) else 'Cache/local', str(local))
    elif MODEL_SOURCE == 'attached_only':
        raise FileNotFoundError('Anexe o peso: ' + spec['target'])
    else:
        missing.append(spec)

catalogs = {}
needed = 0
for spec in missing:
    key = (spec['repo'], spec['revision'])
    if key not in catalogs:
        catalogs[key] = HfApi(token=token).model_info(spec['repo'], revision=spec['revision'], files_metadata=True)
    info = catalogs[key]
    entry = next((item for item in info.siblings if item.rfilename == spec['file']), None)
    if entry is None or entry.size is None:
        raise FileNotFoundError('Arquivo/tamanho não publicado: ' + spec['repo'] + '/' + spec['file'])
    spec['revision'] = info.sha
    spec['size'] = entry.size
    needed += entry.size
if missing:
    free = shutil.disk_usage(WORK_ROOT).free
    if needed + 5 * 1024**3 > free:
        raise RuntimeError(f'Precisamos de {needed/1024**3:.1f} GiB + 5 GiB de reserva em {WORK_ROOT}; '
                           f'há {free/1024**3:.1f} GiB. Anexe os pesos como Kaggle Model antes de baixar.')
    for spec in missing:
        # Own subdirectory prevents cache collisions between different repositories.
        download_root = MODEL_CACHE if spec['repo'] == OFFICIAL_REPO else MODEL_CACHE / 'singularity-source'
        downloaded = Path(hf_hub_download(repo_id=spec['repo'], filename=spec['file'],
                                         revision=spec['revision'], local_dir=download_root, token=token))
        check_weight(downloaded, spec['size'])
        safe_link(downloaded, MODELS_DIR / spec['target'])
        sources[spec['target']] = ('Download temporário', str(downloaded))
total = 0
for rel in REQUIRED_FILES:
    size = check_weight(MODELS_DIR / rel)
    total += size
    print(rel, f'{size/1024**3:.2f} GiB', sources[rel][0])
print(f'Total: {total/1024**3:.2f} GiB; Singularity Pruned + encoder + VAEs + uma LoRA v0.1.')


## ⑦ Opcional: publicar os pesos como Kaggle Model persistente

Use esta célula **uma única vez**, depois que a célula anterior tiver baixado todos os pesos.

Ela usa `kagglehub.model_upload(...)`. O Model ficará na sua conta do Kaggle; depois você pode anexá-lo ao notebook via **Add Input / Models**.  
Depois disso, volte `PUBLISH_WEIGHTS_TO_KAGGLE_MODEL = False`.

> Recomendo conferir na página do Model se a visibilidade está como **Private** antes de guardar conteúdo que você não quer tornar público.


In [ ]:
if not PUBLISH_WEIGHTS_TO_KAGGLE_MODEL:
    print("Publicação desativada. Para publicar, volte à configuração e coloque:")
    print("PUBLISH_WEIGHTS_TO_KAGGLE_MODEL = True")
else:
    import kagglehub

    # Garante que o diretório de staging tenha exatamente os arquivos necessários.
    publish_dir = WORK_ROOT / "publish_model"
    if publish_dir.exists():
        shutil.rmtree(publish_dir)
    publish_dir.mkdir(parents=True)

    for rel in REQUIRED_FILES:
        src = MODELS_DIR / rel
        if not src.exists():
            raise FileNotFoundError(rel)
        dst = publish_dir / rel
        dst.parent.mkdir(parents=True, exist_ok=True)

        # model_upload precisa de arquivos reais; symlinks para /kaggle/input não são necessários.
        # Esta célula é pensada para a PRIMEIRA sessão, quando os arquivos estão em MODEL_CACHE.
        real_src = src.resolve()
        if "/kaggle/input/" in str(real_src):
            print(f"[já persistente] {rel} -> {real_src}")
            continue

        # Hardlink evita duplicar bytes no mesmo filesystem; fallback para symlink/cópia se necessário.
        try:
            os.link(real_src, dst)
        except Exception:
            try:
                dst.symlink_to(real_src)
            except Exception:
                shutil.copy2(real_src, dst)

    real_files = [p for p in publish_dir.rglob("*") if p.is_file()]
    if not real_files:
        print("Nada para publicar: os pesos já parecem estar vindo de um Kaggle Model anexado.")
    else:
        print("Publicando para:", KAGGLE_MODEL_HANDLE)
        print("Arquivos:", len(real_files))
        kagglehub.model_upload(
            KAGGLE_MODEL_HANDLE,
            str(publish_dir),
            version_notes="MiniMax H3 Ref2VA para ComfyUI/Kaggle T4x2"
        )
        print("\nUpload solicitado com sucesso.")
        print("Agora abra a página do Model no Kaggle, confirme que está PRIVATE e, em novas sessões, use Add Input para anexá-lo.")


## ⑧ Referências persistentes em Kaggle Dataset

Guarde imagens, vídeos e áudios de referência em um **Private Dataset**.  
Depois anexe o Dataset com **Add Input** e defina `ASSET_DATASET_HINT` na configuração.

Esta célula cria links em `ComfyUI/input/kaggle_assets/`, sem copiar os arquivos.


In [ ]:
from pathlib import Path

ASSET_EXTS = {
    ".png", ".jpg", ".jpeg", ".webp", ".bmp",
    ".mp4", ".mov", ".webm", ".mkv",
    ".wav", ".mp3", ".flac", ".m4a", ".ogg",
}

asset_out = COMFY_HOME / "input" / "kaggle_assets"
asset_out.mkdir(parents=True, exist_ok=True)

def choose_asset_roots():
    if not INPUT_ROOT.exists():
        return []
    children = [p for p in INPUT_ROOT.iterdir() if p.is_dir()]
    if ASSET_DATASET_HINT:
        return [p for p in children if ASSET_DATASET_HINT.lower() in p.name.lower()]
    return []

roots = choose_asset_roots()

if not roots:
    print("Nenhum Dataset de referências selecionado.")
    print("Diretórios atuais em /kaggle/input:")
    if INPUT_ROOT.exists():
        for p in sorted(INPUT_ROOT.iterdir()):
            if p.is_dir():
                print(" -", p.name)
    print("\nDefina ASSET_DATASET_HINT no início do notebook com parte do nome do seu Dataset.")
else:
    linked = 0
    for root in roots:
        for f in root.rglob("*"):
            if not f.is_file() or f.suffix.lower() not in ASSET_EXTS:
                continue

            rel = f.relative_to(root)
            # prefixo evita colisões entre datasets
            dst = asset_out / root.name / rel
            dst.parent.mkdir(parents=True, exist_ok=True)
            if dst.exists() or dst.is_symlink():
                continue
            dst.symlink_to(f)
            linked += 1

    print(f"{linked} arquivos de referência ligados em:")
    print(asset_out)
    print("Eles aparecerão dentro do diretório kaggle_assets no ComfyUI.")


## ⑨ Iniciar ComfyUI e criar o túnel público

O servidor inicia em `--lowvram`.

Se a versão do ComfyUI oferecer as flags, o notebook também ativa:
- `--fast-disk`
- `--disable-pinned-memory`
- `--reserve-vram 0.8`

**Atenção:** o link `trycloudflare.com` é público. Não o compartilhe.


In [ ]:
import os
import signal
import subprocess
import sys
import time
import re

import requests
from IPython.display import HTML, display

# 兜底
if "TUNNEL_METHOD" not in globals():
    TUNNEL_METHOD = "cloudflared"

BASE_URL = f"http://127.0.0.1:{PORT}"
PID_FILE = WORK_ROOT / "server.pid"
TUNNEL_PID_FILE = WORK_ROOT / "tunnel.pid"

# 重跑本格时先停掉旧进程
if PID_FILE.exists():
    try:
        os.kill(int(PID_FILE.read_text().strip()), signal.SIGTERM)
        time.sleep(2)
    except Exception:
        pass
    PID_FILE.unlink(missing_ok=True)
if TUNNEL_PID_FILE.exists():
    try:
        os.kill(int(TUNNEL_PID_FILE.read_text().strip()), signal.SIGTERM)
        time.sleep(1)
    except Exception:
        pass
    TUNNEL_PID_FILE.unlink(missing_ok=True)

env = os.environ.copy()
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
env["HF_HOME"] = str(CACHE_HOME)

args = [
    sys.executable, "main.py",
    "--listen", "127.0.0.1",
    "--port", str(PORT),
    "--lowvram",
    "--preview-method", "none",
    "--output-directory", str(GEN_DIR),
    "--temp-directory", str(TMP_DIR),
]
help_out = subprocess.run(
    [sys.executable, "main.py", "--help"],
    cwd=COMFY_HOME, env=env, capture_output=True, text=True,
).stdout
if "--reserve-vram" in help_out:
    args += ["--reserve-vram", "0.8"]
if "--fast-disk" in help_out:
    args += ["--fast-disk"]
if "--disable-pinned-memory" in help_out:
    args += ["--disable-pinned-memory"]

print("$", " ".join(args))
log_fp = open(LOG_FILE, "w", encoding="utf-8")
SERVER = subprocess.Popen(
    args, cwd=COMFY_HOME, env=env,
    stdout=log_fp, stderr=subprocess.STDOUT,
)
PID_FILE.write_text(str(SERVER.pid))

t0 = time.time()
while time.time() - t0 < 300:
    if SERVER.poll() is not None:
        log_fp.flush()
        raise RuntimeError("ComfyUI 启动即退出，日志尾部：\n" + log_tail(120))
    try:
        if requests.get(f"{BASE_URL}/system_stats", timeout=3).ok:
            print(f"ComfyUI 已就绪（耗时 {time.time() - t0:.0f}s）")
            break
    except Exception:
        pass
    time.sleep(2)
else:
    raise RuntimeError("ComfyUI 300 秒内未就绪：\n" + log_tail(120))

# ── 双卡自检 ──
try:
    _stats = requests.get(f"{BASE_URL}/system_stats", timeout=5).json()
    _devs = [d for d in _stats.get("devices", []) if d.get("type") == "cuda"]
    print(f"ComfyUI 识别到 {len(_devs)} 张 CUDA 显卡：")
    for d in _devs:
        print(f"  {d.get('name')} · {d.get('vram_total', 0) / 1024 ** 3:.1f} GiB")
    if len(_devs) >= 2:
        print("✔ 双卡已就位。分工：Dual-T4 Text Encoder 指 cuda:1；两个 VAE 留 cuda:0。")
    else:
        print("⚠️ ComfyUI 只看到 1 张卡，双卡节点将退化为单卡运行。")
except Exception as _exc:
    print("双卡自检失败（不影响使用）:", _exc)

_log_txt = log_tail(200)
if "You need pytorch with cu130" in _log_txt:
    print("⚠️ 日志提示 torch 仍非 cu130：快速量化内核未启用（每步会很慢）。")
    print("   可在 ③ 格末尾看到子进程验证结果；搭配 R2V Turbo LoRA 4-step工作流弥补速度。")
elif "comfy_kitchen backend cuda" in _log_txt and "disabled': True" in _log_txt:
    print("⚠️ comfy-kitchen CUDA 后端被禁用（torch < cu130），速度偏慢但能跑。")
    print("   建议用 turbo4 工作流（8 步蒸馏）弥补速度差距。")
else:
    print("✔ 未见 cu130 禁用警告——快速内核可用时采样会明显更快。")


# ── 公网穿透 ──
PUBLIC_URL = None
TUNNEL_PROC = None

if TUNNEL_METHOD == "cloudflare-named":
    # Cloudflare 命名隧道（自定义域名，不 403）
    # 需要在 Kaggle Secrets 配置：
    #   CLOUDFLARE_TUNNEL_TOKEN — Cloudflare Zero Trust → Networks → Tunnels → 创建隧道后获得的 token
    #   CLOUDFLARE_TUNNEL_URL  — 在隧道设置里配置的公网域名（如 https://comfyui.yourdomain.com）
    print("启动 Cloudflare 命名隧道 …")
    CF_BIN = WORK_ROOT / "cloudflared"
    if not CF_BIN.exists():
        import urllib.request
        _cf_url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
        print(f"  下载 cloudflared …")
        urllib.request.urlretrieve(_cf_url, str(CF_BIN))
        os.chmod(str(CF_BIN), 0o755)
    # 从 Kaggle Secrets 读取 token 和 URL
    _cf_token = ""
    _cf_tunnel_url = ""
    try:
        from kaggle_secrets import UserSecretsClient
        _secrets = UserSecretsClient()
        _cf_token = _secrets.get_secret("CLOUDFLARE_TUNNEL_TOKEN") or ""
        _cf_tunnel_url = _secrets.get_secret("CLOUDFLARE_TUNNEL_URL") or ""
    except Exception:
        pass
    if not _cf_token:
        raise RuntimeError(
            "需要 CLOUDFLARE_TUNNEL_TOKEN。\n"
            "设置步骤：\n"
            "  1. 登录 https://one.dash.cloudflare.com → Networks → Tunnels → Create a tunnel\n"
            "  2. 选择 Cloudflared，取个名字，下一步会显示 install 命令，复制其中的 --token 值\n"
            "  3. 在 Public Hostname 里添加一条：subdomain.yourdomain.com → http://localhost:%d\n"
            "  4. Kaggle Add-ons → Secrets 添加 CLOUDFLARE_TUNNEL_TOKEN 和 CLOUDFLARE_TUNNEL_URL" % PORT
        )
    if not _cf_tunnel_url:
        raise RuntimeError(
            "需要 CLOUDFLARE_TUNNEL_URL（你在 Cloudflare 隧道里配置的公网域名，如 https://comfyui.yourdomain.com）"
        )
    _cf_tunnel_url = _cf_tunnel_url.strip().rstrip("/")
    if not _cf_tunnel_url.startswith("http"):
        _cf_tunnel_url = "https://" + _cf_tunnel_url
    # 启动命名隧道
    _tunnel_log = open(WORK_ROOT / "cloudflared.log", "w", encoding="utf-8")
    TUNNEL_PROC = subprocess.Popen(
        [str(CF_BIN), "--no-autoupdate", "tunnel", "run", "--token", _cf_token],
        stdout=_tunnel_log, stderr=subprocess.STDOUT,
    )
    TUNNEL_PID_FILE.write_text(str(TUNNEL_PROC.pid))
    # 命名隧道连接较快，等几秒确认进程存活
    time.sleep(5)
    if TUNNEL_PROC.poll() is not None:
        _err_log = (WORK_ROOT / "cloudflared.log").read_text(encoding="utf-8", errors="replace")
        raise RuntimeError(f"cloudflared 启动即退出，日志：\n{_err_log[-500:]}")
    PUBLIC_URL = _cf_tunnel_url
    print(f"✔ Cloudflare 命名隧道就绪：{PUBLIC_URL}")

elif TUNNEL_METHOD == "cloudflared":
    print("启动 cloudflared 隧道（免注册免 token）…")
    CF_BIN = WORK_ROOT / "cloudflared"
    if not CF_BIN.exists():
        import urllib.request
        _cf_url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
        print(f"  下载 cloudflared …")
        urllib.request.urlretrieve(_cf_url, str(CF_BIN))
        os.chmod(str(CF_BIN), 0o755)
    # cloudflared 输出 URL 到 stderr
    _tunnel_log = open(WORK_ROOT / "cloudflared.log", "w", encoding="utf-8")
    TUNNEL_PROC = subprocess.Popen(
        [str(CF_BIN), "--no-autoupdate", "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
        stdout=_tunnel_log, stderr=subprocess.STDOUT,
    )
    TUNNEL_PID_FILE.write_text(str(TUNNEL_PROC.pid))
    # 从日志里抓取 trycloudflare.com URL
    _cf_t0 = time.time()
    while time.time() - _cf_t0 < 60:
        if TUNNEL_PROC.poll() is not None:
            raise RuntimeError("cloudflared 启动即退出，查看 cloudflared.log")
        time.sleep(3)
        try:
            _cf_log = (WORK_ROOT / "cloudflared.log").read_text(encoding="utf-8", errors="replace")
            _urls = re.findall(r'https://[\w-]+\.trycloudflare\.com', _cf_log)
            if _urls:
                PUBLIC_URL = _urls[0]
                break
        except Exception:
            pass
    if not PUBLIC_URL:
        raise RuntimeError("cloudflared 60 秒内未返回公网 URL，查看 cloudflared.log")
    print(f"✔ cloudflared 隧道就绪")

elif TUNNEL_METHOD == "ngrok":
    print("启动 ngrok 隧道 …")
    from kaggle_secrets import UserSecretsClient
    from pyngrok import ngrok

    _token = UserSecretsClient().get_secret("NGROK_TOKEN")
    if not _token:
        raise RuntimeError("读不到 NGROK_TOKEN：请在 Add-ons → Secrets 里添加后重跑本格，或改用 TUNNEL_METHOD=\"cloudflared\"。")
    ngrok.set_auth_token(_token)
    _tunnel = ngrok.connect(PORT, "http")
    PUBLIC_URL = _tunnel.public_url
    print(f"✔ ngrok 隧道就绪")

elif TUNNEL_METHOD == "auto":
    # 优先命名隧道（有 token），再快速隧道，最后 ngrok
    _auto_token = ""
    _auto_url = ""
    try:
        from kaggle_secrets import UserSecretsClient
        _secrets = UserSecretsClient()
        _auto_token = _secrets.get_secret("CLOUDFLARE_TUNNEL_TOKEN") or ""
        _auto_url = _secrets.get_secret("CLOUDFLARE_TUNNEL_URL") or ""
    except Exception:
        pass

    # 1) 命名隧道
    if _auto_token and _auto_url:
        try:
            print("auto: 尝试 Cloudflare 命名隧道 …")
            CF_BIN = WORK_ROOT / "cloudflared"
            if not CF_BIN.exists():
                import urllib.request
                urllib.request.urlretrieve(
                    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
                    str(CF_BIN))
                os.chmod(str(CF_BIN), 0o755)
            _auto_url = _auto_url.strip().rstrip("/")
            if not _auto_url.startswith("http"):
                _auto_url = "https://" + _auto_url
            _tunnel_log = open(WORK_ROOT / "cloudflared.log", "w", encoding="utf-8")
            TUNNEL_PROC = subprocess.Popen(
                [str(CF_BIN), "--no-autoupdate", "tunnel", "run", "--token", _auto_token],
                stdout=_tunnel_log, stderr=subprocess.STDOUT)
            TUNNEL_PID_FILE.write_text(str(TUNNEL_PROC.pid))
            time.sleep(5)
            if TUNNEL_PROC.poll() is not None:
                raise RuntimeError("cloudflared 启动即退出")
            PUBLIC_URL = _auto_url
            TUNNEL_METHOD = "cloudflare-named"
            print(f"\u2714 Cloudflare 命名隧道就绪（auto）: {PUBLIC_URL}")
        except Exception as _named_err:
            print(f"命名隧道失败（{_named_err}），尝试快速隧道 …")

    # 2) 快速隧道
    if not PUBLIC_URL:
        try:
            print("auto: 尝试 cloudflared 快速隧道 …")
            CF_BIN = WORK_ROOT / "cloudflared"
            if not CF_BIN.exists():
                import urllib.request
                urllib.request.urlretrieve(
                    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
                    str(CF_BIN))
                os.chmod(str(CF_BIN), 0o755)
            _tunnel_log = open(WORK_ROOT / "cloudflared.log", "w", encoding="utf-8")
            TUNNEL_PROC = subprocess.Popen(
                [str(CF_BIN), "--no-autoupdate", "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
                stdout=_tunnel_log, stderr=subprocess.STDOUT)
            TUNNEL_PID_FILE.write_text(str(TUNNEL_PROC.pid))
            _cf_t0 = time.time()
            while time.time() - _cf_t0 < 60:
                if TUNNEL_PROC.poll() is not None:
                    raise RuntimeError("cloudflared 启动即退出")
                time.sleep(3)
                try:
                    _cf_log = (WORK_ROOT / "cloudflared.log").read_text(encoding="utf-8", errors="replace")
                    _urls = re.findall(r'https://[\w-]+\.trycloudflare\.com', _cf_log)
                    if _urls:
                        PUBLIC_URL = _urls[0]
                        break
                except Exception:
                    pass
            if not PUBLIC_URL:
                raise RuntimeError("cloudflared 60 秒内未返回 URL")
            TUNNEL_METHOD = "cloudflared"
            print(f"\u2714 cloudflared 快速隧道就绪（auto）: {PUBLIC_URL}")
        except Exception as _cf_err:
            print(f"快速隧道失败（{_cf_err}），回退到 ngrok …")

    # 3) ngrok 兜底
    if not PUBLIC_URL:
        if TUNNEL_PROC and TUNNEL_PROC.poll() is None:
            TUNNEL_PROC.terminate()
        from kaggle_secrets import UserSecretsClient
        from pyngrok import ngrok
        try:
            _token = UserSecretsClient().get_secret("NGROK_TOKEN")
        except Exception:
            _token = ""
        if not _token:
            raise RuntimeError("所有隧道均失败且 ngrok 无 NGROK_TOKEN")
        ngrok.set_auth_token(_token)
        _tunnel = ngrok.connect(PORT, "http")
        PUBLIC_URL = _tunnel.public_url
        TUNNEL_METHOD = "ngrok"
        print(f"\u2714 ngrok 隧道就绪（auto 回退）: {PUBLIC_URL}")

else:
    raise ValueError(f"未知 TUNNEL_METHOD: {TUNNEL_METHOD}")

display(HTML(f"""
<div style="
    border:2px solid #3ac97a;
    border-left:12px solid #2bff88;
    border-radius:12px;
    padding:18px 24px;
    margin:14px 0;
    background:rgba(43,217,130,0.10);
    font-family:system-ui,'Segoe UI',sans-serif;
">
  <div style="font-size:20px;font-weight:800;">ComfyUI 已上线</div>
  <div style="margin-top:8px;font-size:16px;">
    公网地址：<a href="{PUBLIC_URL}" target="_blank"><b>{PUBLIC_URL}</b></a>
  </div>
  <div style="margin-top:6px;opacity:.75;">
    打开后：左侧「工作流」面板双击 dual_t4_minimax_h3_r2v_turbo4，无需手改模板
  </div>
</div>
"""))

# ── 保活循环：中断（■）即关停 ──
print("服务保活中……（网页可正常用；点 ■ 中断本格 = 关停服务）")
try:
    while SERVER.poll() is None:
        time.sleep(30)
        print(f"[{time.strftime('%H:%M:%S')}] 运行中 · {PUBLIC_URL}   ", end="\r")
    print("\nComfyUI 进程意外退出，日志尾部：")
    print(log_tail(120))
except KeyboardInterrupt:
    print("\n收到中断，正在关停……")
finally:
    # 关闭隧道
    if TUNNEL_METHOD == "ngrok":
        try:
            from pyngrok import ngrok
            ngrok.disconnect(PUBLIC_URL)
            ngrok.kill()
        except Exception:
            pass
    elif TUNNEL_PROC and TUNNEL_PROC.poll() is None:
        TUNNEL_PROC.terminate()
        try:
            TUNNEL_PROC.wait(timeout=10)
        except Exception:
            TUNNEL_PROC.kill()
    # 关闭 ComfyUI
    if SERVER.poll() is None:
        SERVER.terminate()
        try:
            SERVER.wait(timeout=10)
        except Exception:
            SERVER.kill()
    PID_FILE.unlink(missing_ok=True)
    TUNNEL_PID_FILE.unlink(missing_ok=True)
    log_fp.flush()
    print("已关停。成片仍在", GEN_DIR, "，需要的话先跑「抢救成片」那格再结束会话。")


## ⑩ Usar o Singularity Pruned

Abra **Workflows → `singularity_pruned_t4x2`**.
O alias `dual_t4_minimax_h3_r2v_turbo4` também foi preservado para a célula de abertura existente.

Primeiro teste: 0.2 MP, 5 s, uma imagem enviada pelo LoadImage, referência `match`, **6 passos**.
O nome da LoRA contém `4step`; isso não significa que o workflow esteja limitado a quatro passos.
Para pele, prefira um rosto grande no enquadramento e luz natural. Depois de validar memória,
teste 0.3/0.4 MP. Não ative as LoRAs de 8 passos ou EMA junto à 4step v0.1 do Singularity.

O template tem duas imagens de exemplo: substitua-as no ComfyUI ou desconecte a segunda referência
se usar somente uma. Use `<Picture 1>` e `<Picture 2>` no prompt na ordem em que forem conectadas.
A consulta ao catálogo, a instalação e a abertura não equivalem a uma geração de vídeo validada.

Pesos e temporários continuam em `/kaggle/temp` ou `/kaggle/input`.
Salve os vídeos em `/kaggle/working` pela célula abaixo antes de encerrar.

Fontes: [autor do modelo](https://github.com/AIGC-Singularity/Minimax-h3_Singularity),
[distribuição dos pesos](https://huggingface.co/WarmBloodAban/Minimax-h3_Singularity).


## ⑪ Salvar os vídeos antes de encerrar a sessão

Pare a célula do servidor/túnel e rode esta célula.  
Os vídeos serão copiados para `/kaggle/working/videos`.


In [ ]:
import shutil
from pathlib import Path

VIDEO_EXT = {".mp4", ".webm", ".mov", ".mkv"}
RESCUE_DIR = FINAL_DIR / "videos"
RESCUE_DIR.mkdir(parents=True, exist_ok=True)

found = [p for p in GEN_DIR.rglob("*") if p.is_file() and p.suffix.lower() in VIDEO_EXT]
for f in found:
    shutil.copy2(f, RESCUE_DIR / f.name)

print(f"已复制 {len(found)} 个视频到 {RESCUE_DIR}")
for f in sorted(RESCUE_DIR.iterdir()):
    print(f"  {f.name}\t{f.stat().st_size / 1024**2:.1f} MiB")

## ⑫ Diagnóstico

Use se ocorrer OOM, erro de nó ou falha no túnel.


In [ ]:
import os
import signal
import subprocess

subprocess.run(["nvidia-smi"], check=False)

print("\n—— ComfyUI 日志尾部 ——")
print(log_tail(150))

# 如果服务还挂着想手动关停，取消下面两行注释：
# os.kill(int(PID_FILE.read_text().strip()), signal.SIGTERM)
# print("已发送停止信号")